# 🎓 Przykładowy Projekt Zaliczeniowy
## Analiza i Predykcja Rezygnacji Subskrybentów Platformy Streamingowej Muzyki 🎧

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/example/projekt-przyklad-streaming-muzyka.ipynb)

---

| | |
|---|---|
| **Autor** | Jan Kowalski |
| **Nr indeksu** | 123456 |
| **Kurs** | Python w Zastosowaniach Big Data (studia stacjonarne) |
| **Semestr** | Zimowy 2026/2027 |
| **Data** | Styczeń 2027 |

---

## 🎯 Cel projektu

Celem projektu jest zbadanie **czynników wpływających na rezygnację subskrybentów (churn)** platformy streamingowej muzyki oraz budowa modelu predykcyjnego, który — na podstawie zachowań i cech słuchacza — przewidzi, czy dany użytkownik **zrezygnuje z subskrypcji**.

**Pytanie badawcze:**
> *Które cechy użytkownika (typ subskrypcji, aktywność słuchania, liczba playlist, kontakt z supportem) mają największy wpływ na prawdopodobieństwo rezygnacji?*

## 🛠️ Technologie

| Technologia | Zastosowanie |
|---|---|
| **Apache Spark (PySpark)** | Przetwarzanie i analiza dużych zbiorów danych |
| **Spark SQL** | Zaawansowane zapytania i agregacje |
| **MLlib (Random Forest)** | Klasyfikacja — prognozowanie rezygnacji (churn) |
| **Matplotlib / Seaborn** | Wizualizacje i wykresy |

## 📋 Spis treści

1. Konfiguracja środowiska
2. Generowanie i ładowanie danych
3. Eksploracyjna Analiza Danych (EDA)
4. Analiza z Spark SQL
5. Feature Engineering
6. Modelowanie ML (Random Forest)
7. Ewaluacja modelu
8. Wnioski i rekomendacje

---

> 💡 **Uwaga dla studentów:** Ten notebook to **przykładowy** projekt zaliczeniowy — ilustruje oczekiwaną strukturę, poziom szczegółowości i sposób prezentacji wyników. Twój projekt powinien dotyczyć **innego tematu** i **własnego zbioru danych**!


---
## 1. Konfiguracja środowiska 🔧

Instalujemy i importujemy wszystkie niezbędne biblioteki. Na początku każdego projektu warto zebrać wszystkie importy w jednym miejscu — ułatwia to debugowanie i późniejsze uruchamianie.


In [ ]:
# @title 🔧 Instalacja i importy
!pip install pyspark -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Ustawienia wizualizacji
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams["figure.dpi"] = 110

print("✅ Wszystkie biblioteki załadowane pomyślnie!")


In [ ]:
# @title ⚡ Inicjalizacja SparkSession
spark = (
    SparkSession.builder
    .appName("Analiza Rezygnacji Subskrybentow Streamingu Muzyki")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"✅ Apache Spark {spark.version} uruchomiony!")
print(f"🖥️  Tryb: {spark.sparkContext.master}")
print(f"🔢 Domyślna liczba partycji: {spark.conf.get('spark.sql.shuffle.partitions')}")


---
## 2. Generowanie i ładowanie danych 📊

W tym projekcie korzystamy z **syntetycznego zbioru danych** inspirowanego rzeczywistymi danymi platform streamingowych muzyki (np. Spotify, Deezer). Zbiór zawiera **~400 000 rekordów**, co czyni go odpowiednim kandydatem do przetwarzania Big Data.

**Opis kolumn:**

| Kolumna | Typ | Opis |
|---|---|---|
| `user_id` | Int | Unikalny identyfikator subskrybenta |
| `wiek` | Int | Wiek subskrybenta (16–70 lat) |
| `kraj` | String | Kraj subskrybenta |
| `typ_subskrypcji` | String | Typ subskrypcji: `free`, `premium`, `family` |
| `utwory_tydzien` | Int | Liczba odsłuchanych utworów w tygodniu |
| `liczba_playlist` | Int | Liczba utworzonych playlist |
| `dni_od_ostatniej_sesji` | Int | Liczba dni od ostatniej sesji w aplikacji |
| `zgloszenia_supportu` | Int | Liczba zgłoszeń do supportu w ostatnim kwartale |
| `churn` | Int | **Zmienna docelowa**: 1 = użytkownik zrezygnował z subskrypcji, 0 = pozostał aktywny |


In [ ]:
# @title 🏗️ Generowanie syntetycznych danych
np.random.seed(42)
N = 400_000  # 400 tysięcy rekordów

countries      = ['Polska', 'Niemcy', 'USA', 'Wielka Brytania', 'Hiszpania', 'Brazylia', 'Indie', 'Francja']
country_probs  = [0.18, 0.14, 0.20, 0.12, 0.10, 0.10, 0.09, 0.07]

sub_types  = ['free', 'premium', 'family']
sub_probs  = [0.45, 0.40, 0.15]

user_id         = np.arange(1, N + 1)
wiek            = np.random.randint(16, 71, N)
kraj            = np.random.choice(countries, N, p=country_probs)
typ_subskrypcji = np.random.choice(sub_types, N, p=sub_probs)

# Aktywność zależy od typu subskrypcji (premium/family słuchają więcej)
activity_bias_map = {'free': 0.7, 'premium': 1.2, 'family': 1.1}
activity_mult = np.array([activity_bias_map[t] for t in typ_subskrypcji])

utwory_tydzien = np.clip(
    (np.random.gamma(shape=4.0, scale=8.0, size=N) * activity_mult).astype(int), 0, 400
)
liczba_playlist = np.clip(
    np.random.poisson(lam=3.5 * activity_mult), 0, 40
)
dni_od_ostatniej_sesji = np.clip(
    np.random.exponential(scale=6.0 / activity_mult, size=N).astype(int), 0, 120
)

# Zgłoszenia do supportu — użytkownicy free nieco częściej zgłaszają problemy
zgloszenia_supportu = np.random.poisson(lam=0.3, size=N)
extra_free_tickets  = (typ_subskrypcji == 'free').astype(int) * np.random.binomial(1, 0.15, N)
zgloszenia_supportu = np.clip(zgloszenia_supportu + extra_free_tickets, 0, 15)

# Prawdopodobieństwo rezygnacji (churn) — funkcja logistyczna
z = (
    -3.0
    + 0.05 * dni_od_ostatniej_sesji
    - 0.02 * utwory_tydzien
    - 0.15 * liczba_playlist
    + 0.35 * zgloszenia_supportu
    + np.where(typ_subskrypcji == 'free', 1.0, 0.0)
    - np.where(typ_subskrypcji == 'family', 0.4, 0.0)
    + 0.01 * (wiek - 40)
)
prob_churn = 1 / (1 + np.exp(-z))
churn = np.random.binomial(1, prob_churn)

# Tworzymy pandas DataFrame, a następnie konwertujemy do Spark
pdf = pd.DataFrame({
    'user_id':                user_id.astype('int32'),
    'wiek':                   wiek.astype('int32'),
    'kraj':                   kraj,
    'typ_subskrypcji':        typ_subskrypcji,
    'utwory_tydzien':         utwory_tydzien.astype('int32'),
    'liczba_playlist':        liczba_playlist.astype('int32'),
    'dni_od_ostatniej_sesji': dni_od_ostatniej_sesji.astype('int32'),
    'zgloszenia_supportu':    zgloszenia_supportu.astype('int32'),
    'churn':                  churn.astype('int32'),
})

# Konwersja do Apache Spark DataFrame
df = spark.createDataFrame(pdf)
df.cache()  # Cachujemy w pamięci, bo będziemy wielokrotnie odpytywać

print(f"✅ Wygenerowano {N:,} rekordów")
print(f"📊 Odsetek rezygnacji (churn): {churn.mean()*100:.1f}%")
print(f"🔢 Liczba partycji: {df.rdd.getNumPartitions()}")


In [ ]:
# @title 🔍 Schemat i podgląd danych
print("=" * 50)
print("SCHEMAT DANYCH")
print("=" * 50)
df.printSchema()

print("\n" + "=" * 50)
print("PIERWSZE 5 REKORDÓW")
print("=" * 50)
df.show(5, truncate=False)

print(f"\n📐 Rozmiar zbioru: {df.count():,} wierszy × {len(df.columns)} kolumn")


---
## 3. Eksploracyjna Analiza Danych (EDA) 🔍

Zanim przejdziemy do modelowania, musimy dobrze **poznać nasze dane**. EDA odpowiada na pytania:
- Jak duży jest odsetek rezygnacji (churn) w całej populacji?
- Który typ subskrypcji rezygnuje najczęściej?
- Czy kraj subskrybenta ma wpływ na rezygnację?
- Czy niska aktywność (mało odsłuchanych utworów, dawno niewidziana sesja) koreluje z rezygnacją?


In [ ]:
# @title 📈 Statystyki opisowe
print("=" * 50)
print("STATYSTYKI OPISOWE — zmienne numeryczne")
print("=" * 50)
df.select('wiek', 'utwory_tydzien', 'liczba_playlist', 'dni_od_ostatniej_sesji', 'zgloszenia_supportu').describe().show()

# Rozkład zmiennej docelowej
print("=" * 50)
print("ROZKŁAD ZMIENNEJ DOCELOWEJ (churn)")
print("=" * 50)
df.groupBy('churn').count().withColumn(
    'procent', F.round(F.col('count') / df.count() * 100, 2)
).orderBy('churn').show()


In [ ]:
# @title 💳 Analiza per typ subskrypcji i kraj
sub_stats = (
    df.groupBy('typ_subskrypcji')
    .agg(
        F.count('*').alias('liczba_uzytkownikow'),
        F.round(F.mean('churn') * 100, 1).alias('pct_churn'),
        F.round(F.mean('utwory_tydzien'), 1).alias('sr_utwory_tydzien'),
        F.round(F.mean('dni_od_ostatniej_sesji'), 1).alias('sr_dni_od_sesji'),
    )
    .orderBy(F.desc('pct_churn'))
)
print("Statystyki rezygnacji per typ subskrypcji:")
sub_stats.show()

country_stats = (
    df.groupBy('kraj')
    .agg(
        F.count('*').alias('liczba_uzytkownikow'),
        F.round(F.mean('churn') * 100, 1).alias('pct_churn'),
    )
    .orderBy(F.desc('pct_churn'))
)
print("Statystyki rezygnacji per kraj:")
country_stats.show()

# Konwertujemy do pandas na potrzeby wizualizacji
sub_pdf = sub_stats.toPandas()
country_pdf = country_stats.toPandas()


In [ ]:
# @title 🕒 Aktywność vs. churn — grupy wg dni od ostatniej sesji
activity_stats = (
    df.withColumn(
        'grupa_aktywnosci',
        F.when(F.col('dni_od_ostatniej_sesji') <= 7, '0-7 dni')
         .when(F.col('dni_od_ostatniej_sesji') <= 30, '8-30 dni')
         .when(F.col('dni_od_ostatniej_sesji') <= 60, '31-60 dni')
         .otherwise('60+ dni')
    )
    .groupBy('grupa_aktywnosci')
    .agg(
        F.count('*').alias('liczba_uzytkownikow'),
        F.round(F.mean('churn') * 100, 1).alias('pct_churn'),
    )
    .orderBy('pct_churn')
)
print("Churn wg grup aktywności (dni od ostatniej sesji):")
activity_stats.show()


In [ ]:
# @title 📊 Wizualizacje EDA
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Analiza Rezygnacji Subskrybentów — Eksploracja Danych', fontsize=15, fontweight='bold', y=1.01)

# 1. % churn per typ subskrypcji
ax1 = axes[0, 0]
bars = ax1.bar(
    sub_pdf['typ_subskrypcji'],
    sub_pdf['pct_churn'],
    color=sns.color_palette('husl', len(sub_pdf))
)
ax1.set_title('% Rezygnacji per typ subskrypcji', fontweight='bold')
ax1.set_xlabel('Typ subskrypcji')
ax1.set_ylabel('% churn')
for bar, val in zip(bars, sub_pdf['pct_churn']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

# 2. % churn per kraj
ax2 = axes[0, 1]
country_sorted = country_pdf.sort_values('pct_churn', ascending=True)
ax2.barh(country_sorted['kraj'], country_sorted['pct_churn'], color='steelblue')
ax2.set_title('% Rezygnacji per kraj', fontweight='bold')
ax2.set_xlabel('% churn')

# 3. Rozkład dni od ostatniej sesji (histogram) wg churn
ax3 = axes[1, 0]
sample_pdf = pdf.sample(5000, random_state=42)
ax3.hist(sample_pdf[sample_pdf['churn'] == 0]['dni_od_ostatniej_sesji'], bins=30,
         alpha=0.6, label='Aktywny', color='steelblue')
ax3.hist(sample_pdf[sample_pdf['churn'] == 1]['dni_od_ostatniej_sesji'], bins=30,
         alpha=0.6, label='Zrezygnował', color='coral')
ax3.set_title('Dni od ostatniej sesji wg statusu', fontweight='bold')
ax3.set_xlabel('Dni od ostatniej sesji')
ax3.set_ylabel('Liczba użytkowników (próbka 5k)')
ax3.legend()

# 4. Zgłoszenia do supportu vs churn
ax4 = axes[1, 1]
support_stats = pdf.groupby('zgloszenia_supportu')['churn'].mean().reset_index()
support_stats = support_stats[support_stats['zgloszenia_supportu'] <= 6]
ax4.plot(support_stats['zgloszenia_supportu'], support_stats['churn'] * 100,
         marker='o', linewidth=2, color='coral', markersize=6)
ax4.set_title('% Rezygnacji wg liczby zgłoszeń do supportu', fontweight='bold')
ax4.set_xlabel('Liczba zgłoszeń do supportu')
ax4.set_ylabel('% churn')

plt.tight_layout()
plt.savefig('eda_visualizations.png', dpi=110, bbox_inches='tight')
plt.show()
print("📊 Wykresy zapisane do: eda_visualizations.png")


---
## 4. Analiza z Spark SQL 🗃️

Spark SQL umożliwia pisanie **standardowych zapytań SQL** bezpośrednio na Spark DataFrames — bez konieczności używania syntaksu PySpark. Jest szczególnie przydatny dla osób znających SQL oraz przy złożonych agregacjach i operacjach okienkowych.


In [ ]:
# @title 🔖 Rejestracja widoku tymczasowego
df.createOrReplaceTempView("subskrybenci")
print("✅ Widok 'subskrybenci' zarejestrowany — można teraz pisać SQL!")


In [ ]:
# @title 🏆 SQL: Ranking krajów wg % churn per typ subskrypcji
query_ranking = """
SELECT
    typ_subskrypcji,
    kraj,
    COUNT(*) AS liczba_uzytkownikow,
    ROUND(AVG(churn) * 100, 1) AS pct_churn
FROM subskrybenci
GROUP BY typ_subskrypcji, kraj
HAVING COUNT(*) >= 1000
ORDER BY typ_subskrypcji, pct_churn DESC
"""

print("🏆 Ranking krajów wg % churn per typ subskrypcji:")
spark.sql(query_ranking).show(30)


In [ ]:
# @title 🪟 SQL: Funkcje okienkowe — TOP 3 kraje wg churn per typ subskrypcji
query_window = """
SELECT * FROM (
    SELECT
        typ_subskrypcji,
        kraj,
        pct_churn,
        liczba_uzytkownikow,
        RANK() OVER (PARTITION BY typ_subskrypcji ORDER BY pct_churn DESC) AS rank_w_typie
    FROM (
        SELECT
            typ_subskrypcji, kraj,
            ROUND(AVG(churn) * 100, 1) AS pct_churn,
            COUNT(*) AS liczba_uzytkownikow
        FROM subskrybenci
        GROUP BY typ_subskrypcji, kraj
        HAVING COUNT(*) >= 1000
    )
) WHERE rank_w_typie <= 3
ORDER BY typ_subskrypcji, rank_w_typie
"""

print("🪟 TOP 3 kraje (najwyższy % churn) per typ subskrypcji:")
spark.sql(query_window).show(30)


In [ ]:
# @title 📐 SQL: Segmentacja użytkowników (CTE)
query_segments = """
WITH segmenty AS (
    SELECT
        *,
        CASE
            WHEN utwory_tydzien >= 40 AND dni_od_ostatniej_sesji <= 7  THEN 'Zaangazowany 🔥'
            WHEN dni_od_ostatniej_sesji > 30                           THEN 'Ryzyko odejscia ⚠️'
            ELSE 'Standardowy'
        END AS segment
    FROM subskrybenci
)
SELECT
    segment,
    COUNT(*) AS liczba_uzytkownikow,
    ROUND(AVG(churn) * 100, 1) AS pct_churn,
    ROUND(AVG(zgloszenia_supportu), 2) AS sr_zgloszen_supportu
FROM segmenty
GROUP BY segment
ORDER BY pct_churn DESC
"""

print("📐 Churn wg segmentu zaangażowania użytkownika:")
spark.sql(query_segments).show()


---
## 5. Feature Engineering ⚙️

**Feature engineering** to proces tworzenia nowych, lepszych cech z istniejących danych. Dobry feature engineering często ma **większy wpływ na jakość modelu** niż wybór algorytmu!

Tworzymy:
- `wskaznik_zaangazowania` — połączenie liczby utworów i playlist w jedną miarę aktywności
- `is_nieaktywny` — flaga: czy użytkownik nie logował się od ponad 30 dni?
- `wskaznik_supportu` — liczba zgłoszeń do supportu znormalizowana wiekiem konta (proxy)
- `grupa_wiekowa` — przedział wiekowy użytkownika


In [ ]:
# @title ⚙️ Tworzenie nowych cech
df_feat = df.withColumns({
    # Wskaźnik ogólnego zaangażowania (0-100+)
    'wskaznik_zaangazowania': F.round(
        F.col('utwory_tydzien') * 0.5 + F.col('liczba_playlist') * 5, 1
    ),

    # Flaga nieaktywności
    'is_nieaktywny': F.when(F.col('dni_od_ostatniej_sesji') > 30, 1).otherwise(0),

    # Wskaźnik kontaktu z supportem względem aktywności
    'wskaznik_supportu': F.round(
        F.col('zgloszenia_supportu') / (F.col('utwory_tydzien') + 1), 3
    ),

    # Grupa wiekowa
    'grupa_wiekowa': F.when(F.col('wiek') < 25, 'do 25')
                       .when(F.col('wiek') < 40, '25-39')
                       .when(F.col('wiek') < 55, '40-54')
                       .otherwise('55+'),
})

print("✅ Nowe cechy dodane!")
df_feat.select(
    'typ_subskrypcji', 'wskaznik_zaangazowania', 'is_nieaktywny',
    'wskaznik_supportu', 'grupa_wiekowa', 'churn'
).show(8)


In [ ]:
# @title 🔗 Analiza korelacji cech numerycznych
num_cols = ['wiek', 'utwory_tydzien', 'liczba_playlist', 'dni_od_ostatniej_sesji',
            'zgloszenia_supportu', 'wskaznik_zaangazowania', 'is_nieaktywny',
            'wskaznik_supportu', 'churn']

# Pobieramy próbkę do obliczenia macierzy korelacji
corr_pdf = df_feat.select(num_cols).sample(fraction=0.05, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = corr_pdf.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn_r', center=0, ax=ax, square=True,
            annot_kws={'size': 9})
ax.set_title('Macierz korelacji — cechy numeryczne', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=110, bbox_inches='tight')
plt.show()

# Korelacja z churn
print("\n📊 Korelacja z 'churn' (zmienna docelowa):")
print(corr_matrix['churn'].drop('churn').sort_values(ascending=False).to_string())


---
## 6. Modelowanie ML — Random Forest (MLlib) 🌲

Budujemy model klasyfikacji binarnej, który przewiduje, czy subskrybent zrezygnuje (`churn = 1`) czy pozostanie aktywny (`churn = 0`).

### Dlaczego Random Forest?
- Działa dobrze z **mieszanymi typami cech** (numeryczne + kategoryczne)
- **Odporny na overfitting** — dzięki baggingowi i losowemu wyborowi cech na każdym węźle
- Daje interpretowalne **feature importance** — wiemy, które cechy naprawdę mają znaczenie
- Skaluje się dobrze w MLlib na klastrze Spark

### Pipeline MLlib
Używamy `Pipeline` — sekwencji transformacji zakończonej estymatorem. To **kluczowa dobra praktyka**, ponieważ:
- Zapobiega **data leakage** — `StringIndexer` jest fitowany wyłącznie na zbiorze treningowym, a następnie aplikowany do testowego
- Ułatwia **deployment modelu** — zapisujemy jeden obiekt, który sam wykonuje preprocessing
- Kod jest czysty, modularny i łatwy do modyfikacji


In [ ]:
# @title 🏗️ Budowa Pipeline ML
# Cechy kategoryczne do zaindeksowania
cat_cols = ['kraj', 'typ_subskrypcji', 'grupa_wiekowa']
# Cechy numeryczne
num_cols_ml = ['wiek', 'utwory_tydzien', 'liczba_playlist', 'dni_od_ostatniej_sesji',
               'zgloszenia_supportu', 'wskaznik_zaangazowania', 'is_nieaktywny',
               'wskaznik_supportu']

# Krok 1: StringIndexer — koduje kategorie jako liczby
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid='keep')
    for col in cat_cols
]

# Krok 2: VectorAssembler — scala wszystkie cechy w jeden wektor
feature_cols = num_cols_ml + [f"{col}_idx" for col in cat_cols]
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features',
    handleInvalid='keep'
)

# Krok 3: Random Forest Classifier
rf = RandomForestClassifier(
    labelCol='churn',
    featuresCol='features',
    numTrees=60,
    maxDepth=8,
    seed=42
)

# Składamy Pipeline
pipeline = Pipeline(stages=indexers + [assembler, rf])

print("✅ Pipeline skonfigurowany:")
for i, stage in enumerate(pipeline.getStages()):
    print(f"  Krok {i+1}: {type(stage).__name__}")


In [ ]:
# @title 🚂 Podział danych i trening modelu
# Podział: 80% trening, 20% test
# seed=42 zapewnia powtórzalność — za każdym uruchomieniem ten sam podział
train_df, test_df = df_feat.randomSplit([0.8, 0.2], seed=42)

train_df.cache()
test_df.cache()

print(f"Zbiór treningowy: {train_df.count():,} rekordów")
print(f"Zbiór testowy:    {test_df.count():,} rekordów")

print("\n⏳ Trenuję model Random Forest...")
model = pipeline.fit(train_df)
print("✅ Model wytrenowany!")

# Predykcje na zbiorze testowym
predictions = model.transform(test_df)
predictions.select(
    "typ_subskrypcji", "kraj", "dni_od_ostatniej_sesji", "churn", "prediction",
    F.round(F.col("probability")[1], 3).alias("prob_churn")
).show(8, truncate=False)


---
## 7. Ewaluacja modelu 📐

Oceniamy model na zbiorze testowym, który nie był używany podczas treningu.

**Dlaczego wiele metryk?** Żadna pojedyncza metryka nie mówi całej prawdy:

| Metryka | Co mierzy? | Kiedy ważna? |
|---|---|---|
| **AUC-ROC** | Zdolność modelu do oddzielenia klas niezależnie od progu; 1.0 = idealny, 0.5 = losowy | Zawsze — szczególnie przy niezrównoważonych klasach |
| **Accuracy** | (TP+TN) / wszystkich | Tylko gdy klasy są zrównoważone; **mylące przy imbalanced data!** |
| **Precision** | Z *przewidzianych rezygnacji* — jaki % faktycznie zrezygnował? | Gdy koszt fałszywego alarmu (np. niepotrzebna oferta retencyjna) jest wysoki |
| **Recall** | Z *faktycznych rezygnacji* — jaki % model wykrył? | Gdy koszt przeoczenia (utrata klienta) jest wysoki |
| **F1-Score** | Harmoniczna średnia Precision i Recall | Kompromis między nimi |

> 💡 **Interpretacja macierzy błędów (Confusion Matrix):**
> - **TP** — użytkownik zrezygnował, model poprawnie to przewidział ✅
> - **TN** — użytkownik pozostał aktywny, model poprawnie to przewidział ✅
> - **FP** — użytkownik pozostał aktywny, model błędnie oznaczył jako rezygnację ❌
> - **FN** — użytkownik zrezygnował, model błędnie oznaczył jako aktywnego ❌ (przeoczony churn)


In [ ]:
# @title 📊 Metryki ewaluacji
binary_eval  = BinaryClassificationEvaluator(labelCol='churn', rawPredictionCol='rawPrediction')
multi_eval_a = MulticlassClassificationEvaluator(labelCol='churn', predictionCol='prediction', metricName='accuracy')
multi_eval_p = MulticlassClassificationEvaluator(labelCol='churn', predictionCol='prediction', metricName='weightedPrecision')
multi_eval_r = MulticlassClassificationEvaluator(labelCol='churn', predictionCol='prediction', metricName='weightedRecall')
multi_eval_f = MulticlassClassificationEvaluator(labelCol='churn', predictionCol='prediction', metricName='f1')

auc       = binary_eval.evaluate(predictions)
accuracy  = multi_eval_a.evaluate(predictions)
precision = multi_eval_p.evaluate(predictions)
recall    = multi_eval_r.evaluate(predictions)
f1        = multi_eval_f.evaluate(predictions)

print("=" * 40)
print("  WYNIKI MODELU — Random Forest")
print("=" * 40)
print(f"  AUC-ROC   : {auc:.4f}")
print(f"  Accuracy  : {accuracy:.4f}  ({accuracy*100:.1f}%)")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 40)


In [ ]:
# @title 📊 Macierz błędów (Confusion Matrix)
cm_data = (
    predictions
    .groupBy('churn', 'prediction')
    .count()
    .toPandas()
)

# Budujemy macierz 2x2
cm = np.zeros((2, 2), dtype=int)
for _, row in cm_data.iterrows():
    cm[int(row['churn']), int(row['prediction'])] += row['count']

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['Przewidziany: Aktywny', 'Przewidziany: Rezygnacja'],
    yticklabels=['Faktyczny: Aktywny', 'Faktyczny: Rezygnacja'],
    annot_kws={'size': 14, 'fontweight': 'bold'}
)
ax.set_title('Macierz błędów — Random Forest', fontweight='bold', fontsize=13, pad=15)
ax.set_xlabel('Predykcja modelu', fontsize=11)
ax.set_ylabel('Prawdziwa etykieta', fontsize=11)

tp, tn = cm[1, 1], cm[0, 0]
fp, fn = cm[0, 1], cm[1, 0]
print(f"\nTP (True Positive):  {tp:,}  — rezygnacja, poprawnie wykryta")
print(f"TN (True Negative):  {tn:,}  — aktywny, poprawnie wykryty")
print(f"FP (False Positive): {fp:,}  — aktywny, błędnie oznaczony jako rezygnacja")
print(f"FN (False Negative): {fn:,}  — rezygnacja, błędnie oznaczona jako aktywny")

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=110, bbox_inches='tight')
plt.show()


In [ ]:
# @title 🌲 Ważność cech (Feature Importance)
rf_model = model.stages[-1]  # ostatni krok w pipeline = RandomForestModel
importances = rf_model.featureImportances.toArray()

feat_imp_df = pd.DataFrame({
    'cecha': feature_cols,
    'waznosc': importances
}).sort_values('waznosc', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('viridis', len(feat_imp_df))
bars = ax.barh(feat_imp_df['cecha'], feat_imp_df['waznosc'], color=colors)
ax.set_xlabel('Ważność cechy (Gini Importance)', fontsize=11)
ax.set_title('Ważność cech — Random Forest Classifier', fontweight='bold', fontsize=13)
ax.invert_yaxis()

for bar, val in zip(bars, feat_imp_df['waznosc']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=110, bbox_inches='tight')
plt.show()

print("\n🏆 TOP 5 najważniejszych cech:")
print(feat_imp_df.head(5).to_string(index=False))


---
## 8. Wnioski i rekomendacje 📝

### 🔍 Najważniejsze odkrycia

**1. Największy wpływ na rezygnację mają:**
- **`dni_od_ostatniej_sesji`** — użytkownicy, którzy nie logowali się dłużej niż 30 dni, rezygnują znacznie częściej
- **`zgloszenia_supportu`** — im więcej problemów zgłasza użytkownik, tym wyższe ryzyko odejścia
- **`typ_subskrypcji`** — subskrybenci `free` rezygnują wyraźnie częściej niż `premium` czy `family`

**2. Typ subskrypcji ma ogromne znaczenie:**
- Subskrybenci **free** mają najwyższy odsetek rezygnacji
- Plan **family** jest najbardziej "lepki" — najniższy churn
- Różnica między najgorszym a najlepszym planem sięga kilkunastu punktów procentowych

**3. Aktywność słuchania jako sygnał wczesnego ostrzegania:**
- Spadek liczby odsłuchanych utworów tygodniowo to silny predyktor nadchodzącej rezygnacji
- Liczba playlist koreluje ujemnie z churn — użytkownicy "zainwestowani" w platformę (mają swoje playlisty) rzadziej odchodzą

**4. Kontakt z supportem:**
- Więcej niż 2 zgłoszenia do supportu w kwartale silnie koreluje z odejściem — to sygnał niezadowolenia klienta

### 📊 Ocena modelu

Model Random Forest osiągnął dobre wyniki na tym zbiorze. Warto byłoby porównać z:
- Regresją logistyczną (baseline, bardziej interpretowalny)
- Gradient Boosting (GBT) — często lepszy od RF
- Modelem uwzględniającym historię płatności i dane transakcyjne

### 🚀 Możliwe rozszerzenia

1. **Prawdziwe dane** — skorzystać z publicznie dostępnych zbiorów (np. Kaggle "Music Streaming Churn")
2. **Modelowanie przeżycia (survival analysis)** — przewidywać *kiedy* nastąpi rezygnacja, nie tylko czy
3. **Spark Streaming** — model wykrywający ryzyko churn w czasie rzeczywistym na podstawie zdarzeń z aplikacji
4. **Deployment na GCP** — Google Cloud Dataproc + BigQuery ML + akcje retencyjne (np. automatyczna oferta rabatowa)

### 💡 Refleksja

Projekt pokazał, że Apache Spark sprawdza się doskonale przy analizie dużych zbiorów danych behawioralnych. Operacje, które na 400 000 rekordów w pandas zajęłyby sekundy, przy milionach rekordów byłyby znacznie trudniejsze — Spark skaluje się poziomo i radzi sobie z terabajtami danych z prawdziwych platform streamingowych.


In [ ]:
# @title 📊 Podsumowanie wyników w jednym wykresie
metrics = {'AUC-ROC': auc, 'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1-Score': f1}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(metrics.keys(), metrics.values(), color=sns.color_palette('Set2', 5), edgecolor='white', linewidth=1.5)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Wartość metryki', fontsize=11)
ax.set_title('Podsumowanie wyników — Random Forest (churn)', fontweight='bold', fontsize=13)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Baseline (losowy): 0.5')
ax.legend()
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('model_summary.png', dpi=110, bbox_inches='tight')
plt.show()


In [ ]:
# @title 🛑 Zatrzymanie SparkSession
spark.stop()
print("✅ SparkSession zatrzymana.")
print("\n🎓 Projekt ukończony! Pliki wyjściowe:")
print("  - eda_visualizations.png")
print("  - correlation_matrix.png")
print("  - feature_importance.png")
print("  - confusion_matrix.png")
print("  - model_summary.png")
